# Stress Testing Forecasts

Stress testing evaluates how forecasts behave under **extreme scenarios** and **tail risk events**.
It is a critical tool for risk management, central banking, and financial regulation.

**Key concepts:**
- **Historical stress tests**: Apply shocks calibrated from past crises (2008 GFC, COVID-19)
- **Sensitivity analysis**: Systematically vary shock magnitudes to map the impact surface
- **Reverse stress tests**: Find the shock magnitude that produces a target adverse outcome

**Topics covered:**
1. Defining historical stress scenarios (GFC-like, COVID-like)
2. Running stress tests with shock propagation via IRF
3. Sensitivity analysis with heat maps
4. Reverse stress testing
5. Consolidated stress test report

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.scenarios import SimpleVAR, StressTest, Shock

import sys
sys.path.insert(0, "..")
from utils.helpers import load_us_macro_quarterly, load_macro_brazil

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Defining Stress Scenarios

We calibrate two historical stress scenarios and one hypothetical scenario:

| Scenario | GDP Shock | Unemployment Shock | Inflation Shock | Source |
|----------|-----------|-------------------|-----------------|--------|
| **GFC-like (2008)** | -4.0 pp | +5.0 pp | -1.0 pp | Great Financial Crisis |
| **COVID-like (2020)** | -9.0 pp (recovery after 2Q) | +8.0 pp (temporary) | +0.5 pp | COVID-19 pandemic |
| **Stagflation** | -2.0 pp | +3.0 pp | +4.0 pp | Hypothetical supply shock |

In [ ]:
# Load data and estimate VAR
df = load_us_macro_quarterly()
var_names = ["gdp_growth", "inflation", "fed_funds", "unemployment"]
endog = df[var_names].values

model = SimpleVAR(endog, p_order=2, var_names=var_names)
steps = 8
print(f"VAR({model.p_order}) with {model.k_vars} variables, {endog.shape[0]} obs")
print(f"Residual std devs: {np.sqrt(np.diag(model.sigma_u)).round(3)}")

# Define stress scenarios as dictionaries of shocks
scenarios = {
    "GFC-like (2008)": [
        Shock(variable="gdp_growth", magnitude=-4.0, shock_type="absolute", period=1, duration=4, decay=0.3),
        Shock(variable="unemployment", magnitude=5.0, shock_type="absolute", period=1, duration=6, decay=0.2),
        Shock(variable="inflation", magnitude=-1.0, shock_type="absolute", period=1, duration=3, decay=0.5),
    ],
    "COVID-like (2020)": [
        Shock(variable="gdp_growth", magnitude=-9.0, shock_type="absolute", period=1, duration=2, decay=0.8),
        Shock(variable="unemployment", magnitude=8.0, shock_type="absolute", period=1, duration=3, decay=0.7),
        Shock(variable="inflation", magnitude=0.5, shock_type="absolute", period=2, duration=4, decay=0.3),
    ],
}

print("\nDefined scenarios:")
for name, shocks in scenarios.items():
    print(f"\n  {name}:")
    for s in shocks:
        print(f"    {s.variable}: {s.magnitude:+.1f} ({s.shock_type}), "
              f"period={s.period}, duration={s.duration}, decay={s.decay}")

## 2. Historical Stress Test

We apply the GFC-like and COVID-like shocks to our VAR model and observe how the
shocks propagate through the system via impulse response functions (IRF).

The `StressTest` class:
1. Computes the baseline (no-shock) forecast
2. Converts shock magnitudes to absolute terms
3. Propagates shocks via IRF matrices
4. Returns baseline vs stressed forecasts and impact

In [ ]:
# Run GFC-like stress test
stress_gfc = StressTest(model)
for shock in scenarios["GFC-like (2008)"]:
    stress_gfc.add_shock(
        variable=shock.variable,
        magnitude=shock.magnitude,
        shock_type=shock.shock_type,
        period=shock.period,
        duration=shock.duration,
        decay=shock.decay,
    )
result_gfc = stress_gfc.run(steps=steps, n_draws=500, seed=42)

# Run COVID-like stress test
stress_covid = StressTest(model)
for shock in scenarios["COVID-like (2020)"]:
    stress_covid.add_shock(
        variable=shock.variable,
        magnitude=shock.magnitude,
        shock_type=shock.shock_type,
        period=shock.period,
        duration=shock.duration,
        decay=shock.decay,
    )
result_covid = stress_covid.run(steps=steps, n_draws=500, seed=42)

# Plot both stress tests side by side
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
horizons = np.arange(1, steps + 1)

for col, var in enumerate(var_names):
    # GFC
    ax = axes[0, col]
    ax.plot(horizons, result_gfc.baseline[var].point, "b-o", label="Baseline", markersize=4)
    ax.plot(horizons, result_gfc.stressed[var].point, "r-s", label="GFC Stressed", markersize=4)
    if result_gfc.stressed[var].lower_95 is not None:
        ax.fill_between(horizons, result_gfc.stressed[var].lower_95,
                        result_gfc.stressed[var].upper_95, alpha=0.1, color="red")
    ax.set_title(f"{var}" + (" (GFC)" if col == 0 else ""), fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # COVID
    ax = axes[1, col]
    ax.plot(horizons, result_covid.baseline[var].point, "b-o", label="Baseline", markersize=4)
    ax.plot(horizons, result_covid.stressed[var].point, "r-s", label="COVID Stressed", markersize=4)
    if result_covid.stressed[var].lower_95 is not None:
        ax.fill_between(horizons, result_covid.stressed[var].lower_95,
                        result_covid.stressed[var].upper_95, alpha=0.1, color="red")
    ax.set_title(f"{var}" + (" (COVID)" if col == 0 else ""), fontsize=11)
    ax.set_xlabel("Horizon")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0, 0].set_ylabel("GFC-like", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("COVID-like", fontsize=12, fontweight="bold")
fig.suptitle("Historical Stress Tests: GFC vs COVID", fontsize=14)
fig.tight_layout()
plt.show()

# Print summaries
print(result_gfc.summary())
print("\n")
print(result_covid.summary())

## 3. Sensitivity Analysis

How sensitive are the forecasts to **shock magnitude**? We vary the GDP shock from
1x to 3x the baseline magnitude and build a heat map showing the impact on all
variables at each horizon.

This helps identify **non-linearities** and **threshold effects** in the model.

In [ ]:
# Sensitivity analysis: vary GDP shock magnitude (1x, 1.5x, 2x, 2.5x, 3x)
multipliers = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
base_gdp_shock = -4.0  # GFC-like base magnitude

# Collect max impact for each variable at each multiplier
impact_matrix = {var: [] for var in var_names}

for mult in multipliers:
    st = StressTest(model)
    st.add_shock("gdp_growth", magnitude=base_gdp_shock * mult,
                 shock_type="absolute", period=1, duration=4, decay=0.3)
    result = st.run(steps=steps, n_draws=100, seed=42)

    for var in var_names:
        # Use the full impact profile, take max absolute impact
        max_val, max_h = result.max_impact(var)
        impact_matrix[var].append(max_val)

# Build heat map: rows = variables, columns = multipliers
impact_df = pd.DataFrame(impact_matrix, index=[f"{m}x" for m in multipliers]).T
print("Max Impact by Variable and Shock Multiplier:")
print(impact_df.round(3))

# Heat map visualization
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(impact_df.values, cmap="RdBu_r", aspect="auto",
               vmin=-np.abs(impact_df.values).max(), vmax=np.abs(impact_df.values).max())

ax.set_xticks(range(len(multipliers)))
ax.set_xticklabels([f"{m}x" for m in multipliers])
ax.set_yticks(range(len(var_names)))
ax.set_yticklabels(var_names)
ax.set_xlabel("Shock Multiplier", fontsize=12)
ax.set_title("Sensitivity Analysis: Max Impact of GDP Shock", fontsize=14)

# Annotate cells
for i in range(len(var_names)):
    for j in range(len(multipliers)):
        val = impact_df.values[i, j]
        color = "white" if abs(val) > np.abs(impact_df.values).max() * 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=10)

plt.colorbar(im, ax=ax, label="Impact (pp)")
fig.tight_layout()
plt.show()

# Also show impact profiles across horizons for different multipliers
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors_sens = plt.cm.Reds(np.linspace(0.3, 1.0, len(multipliers)))

for idx, var in enumerate(var_names):
    ax = axes[idx // 2, idx % 2]
    for m_idx, mult in enumerate(multipliers):
        st = StressTest(model)
        st.add_shock("gdp_growth", magnitude=base_gdp_shock * mult,
                     shock_type="absolute", period=1, duration=4, decay=0.3)
        result = st.run(steps=steps, n_draws=100, seed=42)
        ax.plot(horizons, result.impact[var], "-o", color=colors_sens[m_idx],
                label=f"{mult}x", markersize=3, linewidth=1.5)

    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.set_ylabel("Impact (pp)")
    ax.legend(fontsize=8, title="Multiplier")
    ax.grid(True, alpha=0.3)

fig.suptitle("Sensitivity Analysis: Impact Profiles by Shock Multiplier", fontsize=14)
fig.tight_layout()
plt.show()

## 4. Reverse Stress Test

A reverse stress test answers: **"What shock magnitude is needed to cause a specific adverse outcome?"**

For example: "What GDP shock would push unemployment to 10%?"

The `run_reverse` method uses the IRF to analytically compute the required shock magnitude.

In [ ]:
# Reverse stress test: What GDP shock pushes unemployment to 10.0%?
st_reverse = StressTest(model)
result_reverse = st_reverse.run_reverse(
    target_variable="unemployment",
    target_value=10.0,
    shock_variable="gdp_growth",
    steps=steps,
)

# Show results
print("REVERSE STRESS TEST")
print("=" * 50)
print(f"Question: What GDP shock causes unemployment = 10.0%?")
print(f"\nRequired shock: {result_reverse.shocks[0].magnitude:+.2f} pp to gdp_growth")
print(f"Baseline unemployment (Q+1): {result_reverse.baseline['unemployment'].point[0]:.2f}%")
print(f"Stressed unemployment (Q+1): {result_reverse.stressed['unemployment'].point[0]:.2f}%")

# Plot the reverse stress test result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Unemployment: baseline vs stressed
ax = axes[0]
ax.plot(horizons, result_reverse.baseline["unemployment"].point, "b-o", label="Baseline", markersize=4)
ax.plot(horizons, result_reverse.stressed["unemployment"].point, "r-s", label="Stressed", markersize=4)
ax.axhline(10.0, color="darkred", linestyle="--", alpha=0.7, label="Target (10%)")
ax.set_title("Unemployment", fontsize=12)
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Unemployment (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# GDP: baseline vs stressed
ax = axes[1]
ax.plot(horizons, result_reverse.baseline["gdp_growth"].point, "b-o", label="Baseline", markersize=4)
ax.plot(horizons, result_reverse.stressed["gdp_growth"].point, "r-s", label="Stressed", markersize=4)
ax.set_title("GDP Growth (with required shock)", fontsize=12)
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle(f"Reverse Stress Test: GDP shock = {result_reverse.shocks[0].magnitude:+.2f} pp", fontsize=14)
fig.tight_layout()
plt.show()

# Try another reverse: what inflation shock causes fed_funds to reach 8%?
st_reverse2 = StressTest(model)
result_reverse2 = st_reverse2.run_reverse(
    target_variable="fed_funds",
    target_value=8.0,
    shock_variable="inflation",
    steps=steps,
)

print(f"\nReverse test 2: Inflation shock = {result_reverse2.shocks[0].magnitude:+.2f} pp")
print(f"  -> causes fed_funds to reach {result_reverse2.stressed['fed_funds'].point[0]:.2f}%")

## 5. Stress Test Report

Let's generate a consolidated report comparing all stress scenarios, including
impact tables and comparison charts.

In [ ]:
# Consolidated stress test report
print("=" * 70)
print("STRESS TEST REPORT - US MACRO QUARTERLY MODEL")
print("=" * 70)
print(f"Model: VAR({model.p_order}) | Variables: {', '.join(var_names)}")
print(f"Forecast horizon: {steps} quarters")
print(f"Data: us_macro_quarterly.csv ({endog.shape[0]} observations)")

# Impact summary table
report_rows = []
for scenario_name, result in [("GFC-like", result_gfc), ("COVID-like", result_covid)]:
    for var in var_names:
        max_val, max_h = result.max_impact(var)
        report_rows.append({
            "Scenario": scenario_name,
            "Variable": var,
            "Max Impact": f"{max_val:+.3f}",
            "At Horizon": f"Q+{max_h + 1}",
            "Baseline Q+1": f"{result.baseline[var].point[0]:.3f}",
            "Stressed Q+1": f"{result.stressed[var].point[0]:.3f}",
        })

report_df = pd.DataFrame(report_rows)
print("\n--- Max Impact Summary ---")
print(report_df.to_string(index=False))

# Impact comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, var in enumerate(var_names):
    ax = axes[idx // 2, idx % 2]
    ax.bar(horizons - 0.15, result_gfc.impact[var], width=0.3,
           color="darkred", alpha=0.7, label="GFC-like")
    ax.bar(horizons + 0.15, result_covid.impact[var], width=0.3,
           color="darkorange", alpha=0.7, label="COVID-like")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.set_ylabel("Impact (pp)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Stress Test Impact Comparison: GFC vs COVID", fontsize=14)
fig.tight_layout()
plt.show()

# Reverse stress test summary
print("\n--- Reverse Stress Test Summary ---")
print(f"  To push unemployment to 10%: need GDP shock of {result_reverse.shocks[0].magnitude:+.2f} pp")
print(f"  To push fed_funds to 8%:     need inflation shock of {result_reverse2.shocks[0].magnitude:+.2f} pp")

## Exercise 1: Design a commodity shock stress test for Brazil

Load the `macro_brazil.csv` dataset, estimate a VAR model, and design a stress test
simulating a commodity price shock (e.g., large depreciation of the exchange rate
via a shock to `cambio`). Analyze the impact on inflation (`ipca`) and GDP (`pib_mensal`).

In [ ]:
# TODO: Exercise 1
# Hint:
# df_br = load_macro_brazil()
# endog_br = df_br[["ipca", "selic", "cambio", "pib_mensal", "producao_industrial"]].values
# model_br = SimpleVAR(endog_br, p_order=2, var_names=[...])
# st_br = StressTest(model_br)
# st_br.add_shock("cambio", magnitude=3.0, shock_type="std_dev", period=1, duration=4)
# result_br = st_br.run(steps=12)
# result_br.plot_comparison("ipca")

## Exercise 2: Reverse stress test - what causes GDP < -3%?

Using the US macro model, find the unemployment shock that would cause GDP growth
to fall below -3%. Use `run_reverse` with `target_variable="gdp_growth"` and
`target_value=-3.0`.

In [ ]:
# TODO: Exercise 2
# Hint:
# st_rev = StressTest(model)
# result_rev = st_rev.run_reverse(
#     target_variable="gdp_growth",
#     target_value=-3.0,
#     shock_variable="unemployment",
#     steps=8,
# )
# print(f"Required unemployment shock: {result_rev.shocks[0].magnitude:+.2f} pp")
# result_rev.plot_comparison("gdp_growth")